**HW 2/4.** With the call and put pricer, check if Put-Call parity holds in practice.

The call pricer of `1_HW_EXE_2` (NEW VERSION) and the put pricer of `1_HW_EXE_3` (NEW VERSION).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from typing import Dict, List, Union

"""The pricing function of European call option"""
# change # 2
#Delete.    def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float,
#Delete.                               K: Union[float, List[float], np.ndarray]) -> np.ndarray:
# replace by
def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float,
                           K: Union[float, List[float], np.ndarray],
                           return_greeks: bool = False):
# end change # 2
    """
    Black-Scholes pricer of European call option on non-dividend-paying stock

    param r: risk-free interest rate (which is constant)
    param T: time to maturity (in years)
    param S0: initial spot price of the underlying stock
    param sigma: volatility of the underlying stock
    param K: strike price (or prices)
    param return_greeks: if True, return a dict with the price and the greeks
    """
    # check conditions
    assert sigma > 0

    K = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = ( np.log( S0 / K ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    N_d1_vec = norm.cdf(d1_vec)
    N_d2_vec = norm.cdf(d2_vec)

    # change # 1
    #Delete.    return N_d1_vec * S0 - K_vec * np.exp((-1.0)*r*T) * N_d2_vec
    # replace by
    price = N_d1_vec * S0 - K * np.exp((-1.0)*r*T) * N_d2_vec
    # end change # 1

    # change # 2
    if not return_greeks:
        return price
    n_d1_vec = norm.pdf(d1_vec)                  # N'(d1)
    return {
        "price": price,
        "delta": N_d1_vec,
        "gamma": n_d1_vec / (S0 * sigma * T**0.5),
        "vega":  S0 * n_d1_vec * T**0.5,
        "theta": -S0 * n_d1_vec * sigma / (2 * T**0.5) - r * K * np.exp(-r*T) * N_d2_vec,
        "rho":   K * T * np.exp(-r*T) * N_d2_vec,
    }
    # end change # 2

In [ ]:
"""The pricing function of European put option"""
# change # 1
#Delete.    def black_scholes_eur_call(r: float, T: float, S0: float, sigma: float,
#Delete.                               K: Union[float, List[float], np.ndarray]) -> np.ndarray:
# replace by
def black_scholes_eur_put(r: float, T: float, S0: float, sigma: float,
                          K: Union[float, List[float], np.ndarray]) -> np.ndarray:
# end change # 1
    """
    Black-Scholes pricer of European put option on non-dividend-paying stock

    param r: risk-free interest rate (which is constant)
    param T: time to maturity (in years)
    param S0: initial spot price of the underlying stock
    param sigma: volatility of the underlying stock
    param K: strike price (or prices)
    """
    # check conditions
    assert sigma > 0

    K = np.array([K]) if isinstance(K, float) else np.array(K)

    d1_vec = ( np.log( S0 / K ) + ( r + 0.5 * sigma**2 ) * T ) / ( sigma * T**0.5 )
    d2_vec = d1_vec - sigma * T**0.5

    # change # 2
    #Delete.    N_d1_vec = norm.cdf(d1_vec)
    #Delete.    N_d2_vec = norm.cdf(d2_vec)
    # replace by
    N_minus_d1_vec = norm.cdf(-d1_vec)
    N_minus_d2_vec = norm.cdf(-d2_vec)
    # end change # 2

    # change # 3
    #Delete.    return N_d1_vec * S0 - K_vec * np.exp((-1.0)*r*T) * N_d2_vec
    # replace by
    return K * np.exp((-1.0)*r*T) * N_minus_d2_vec - S0 * N_minus_d1_vec
    # end change # 3

**Put-call parity** at $t = 0$ (cell 30): $c - p = S_0 - K e^{-rT}$. We compute both sides for the strikes and maturities of cell 44.

In [ ]:
# put-call parity at t = 0:  c - p = S0 - K e^{-rT}
K_vec = np.arange(10, 30, 0.01)
T_vec = [1.0, 2.0, 5.0]
S0, sigma, r = 20.0, 0.3, 0.05

plt.figure(figsize=(10, 6))
for T in T_vec:
    c = black_scholes_eur_call(r=r, T=T, S0=S0, sigma=sigma, K=K_vec)
    p = black_scholes_eur_put(r=r, T=T, S0=S0, sigma=sigma, K=K_vec)
    left = c - p                              # from the pricers
    right = S0 - K_vec * np.exp(-r * T)       # from the parity formula
    print(f"T = {int(T)}:  max |(c - p) - (S0 - K e^(-rT))| = {np.max(np.abs(left - right)):.1e}")
    plt.plot(K_vec, left, label=f"c - p,  T = {int(T)}")
    plt.plot(K_vec, right, 'k--', linewidth=0.8)
plt.title('Put-call parity: $c - p$ (colour) and $S_0 - Ke^{-rT}$ (black dashed)')
plt.xlabel('Strike price $K$')
plt.ylabel('value')
plt.legend()
plt.grid(True)
plt.show()

The difference is about $10^{-15}$ (rounding of the computer): parity holds. For the BSM formulas it holds exactly, because $N(x) + N(-x) = 1$.